# 05 — Causal attention in PyTorch

We move from the numeric explanation in notebook 04 to a reusable PyTorch module. The code progresses through tensors, Q/K/V projections, a causal mask, and a reusable class.

The input values are synthetic. The projections are randomly initialized: no AtlasIA text is loaded and no model is trained in this notebook.

## Goal and tensor shapes

The module takes an input of shape **(B, T, d_in)**: B sequences, T positions, and d_in features per position. It returns a context vector at each position.

This step implements causal attention only. Multi-head attention, residual connections, normalization, and the MLP will come next.

In [1]:
import math
import torch
from torch import nn

torch.manual_seed(123)

# Synthetic numeric input: B=2 sequences, T=4 positions, d_in=3.
x = torch.tensor([
    [[0.43, 0.15, 0.89], [0.55, 0.87, 0.66],
     [0.57, 0.85, 0.64], [0.22, 0.58, 0.33]],
    [[0.77, 0.25, 0.10], [0.05, 0.80, 0.55],
     [0.31, 0.46, 0.72], [0.62, 0.12, 0.40]],
])

print("x :", tuple(x.shape))  # (B, T, d_in)

C:\Users\sail_\Documents\Codex\2026-10-03\https-github-com-rasbt-llms-from\outputs\darija-llm-lab\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:377: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:82.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


x : (2, 4, 3)


## 1. Project inputs into Q, K, and V

Three trainable linear layers produce queries **Q**, keys **K**, and values **V**. The same layers are applied at every position in the batch.

In [2]:
d_in = x.shape[-1]
d_out = 4

W_query = nn.Linear(d_in, d_out, bias=False)
W_key = nn.Linear(d_in, d_out, bias=False)
W_value = nn.Linear(d_in, d_out, bias=False)

queries = W_query(x)
keys = W_key(x)
values = W_value(x)

print("Q, K, V :", tuple(queries.shape), tuple(keys.shape), tuple(values.shape))

Q, K, V : (2, 4, 4) (2, 4, 4) (2, 4, 4)


## 2. Attention scores and weights

The product QKᵀ compares each position with the others. We divide the scores by √dₖ, then softmax turns each row into weights that sum to 1.

In [3]:
scores = queries @ keys.transpose(-2, -1)
scores = scores / math.sqrt(d_out)
weights = torch.softmax(scores, dim=-1)

print("Scores :", tuple(scores.shape))  # (B, T, T)
print("Weights for sequence 0:\n", weights[0].round(decimals=3))
assert torch.allclose(weights.sum(dim=-1), torch.ones(x.shape[0], x.shape[1]))

Scores : (2, 4, 4)
Weights for sequence 0:
 tensor([[0.2610, 0.2500, 0.2500, 0.2390],
        [0.2430, 0.2520, 0.2530, 0.2520],
        [0.2430, 0.2520, 0.2530, 0.2510],
        [0.2430, 0.2510, 0.2520, 0.2540]], grad_fn=<RoundBackward1>)


## 3. Causal mask

To predict the next token, a position must not see future positions. We replace their scores with −∞ **before** softmax, so they receive zero weight.

In [4]:
batch_size, sequence_length, _ = x.shape
causal_mask = torch.tril(torch.ones(sequence_length, sequence_length, dtype=torch.bool))

masked_scores = scores.masked_fill(~causal_mask, float("-inf"))
causal_weights = torch.softmax(masked_scores, dim=-1)
context = causal_weights @ values

print("Masque :\n", causal_mask.int())
print("Masked weights:\n", causal_weights[0].round(decimals=3))
print("Contexte :", tuple(context.shape))  # (B, T, d_out)
assert torch.count_nonzero(causal_weights.masked_select(~causal_mask[None, :, :])) == 0
assert torch.allclose(causal_weights.sum(dim=-1), torch.ones(batch_size, sequence_length))

Masque :
 tensor([[1, 0, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 1, 0],
        [1, 1, 1, 1]], dtype=torch.int32)
Masked weights:
 tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.4910, 0.5090, 0.0000, 0.0000],
        [0.3250, 0.3370, 0.3380, 0.0000],
        [0.2430, 0.2510, 0.2520, 0.2540]], grad_fn=<RoundBackward1>)
Contexte : (2, 4, 4)


## 4. Put the operations in a class

The mask is a buffer: it moves with the module to the CPU or GPU but is not learned. Dropout is applied to attention weights during training and disabled by `eval()`.

In [5]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout=0.0):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)
        self.dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(context_length, context_length, dtype=torch.bool))
        self.register_buffer("causal_mask", mask)

    def forward(self, x):
        # x : (B, T, d_in)
        _, sequence_length, _ = x.shape
        if sequence_length > self.causal_mask.size(0):
            raise ValueError("Sequence exceeds context_length.")

        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        scores = queries @ keys.transpose(-2, -1)
        scores = scores / math.sqrt(self.d_out)
        allowed = self.causal_mask[:sequence_length, :sequence_length]
        scores = scores.masked_fill(~allowed, float("-inf"))

        weights = torch.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        context = weights @ values
        return context, weights

In [6]:
attention = CausalSelfAttention(d_in=3, d_out=8, context_length=8, dropout=0.1)
attention.eval()
output, weights = attention(x)

print("Sortie :", tuple(output.shape))
print("Weights:", tuple(weights.shape))
assert output.shape == (2, 4, 8)
assert torch.count_nonzero(weights.masked_select(~causal_mask[None, :, :])) == 0

Sortie : (2, 4, 8)
Weights: (2, 4, 4)


## Key points

- Q and K produce the scores; attention weights combine the V values.
- The causal mask blocks future tokens before softmax.
- The batch dimension lets us process several sequences at once.
- This class is not multi-head yet and does not include the MLP or LayerNorm.
- The data and weights are synthetic; nothing has been trained on Darija.

Next we will add multi-head attention, then assemble the Transformer block.